# Used Functions and Models

## Models

In [7]:
import torch
from torch import nn

class IndependentHeadsMLP(nn.Module):
    """MLP with a shared body and independent per-label output heads."""

    def __init__(self, input_dim, hidden_dim, n_hidden, n_labels, activation="ReLU"):
        super().__init__()
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList(
            [nn.Linear(hidden_dim, hidden_dim) for _ in range(n_hidden)]
        )
        self.heads = nn.ModuleList(
            [nn.Linear(hidden_dim, 1) for _ in range(n_labels)]
        )

        self.activation = nn.ReLU() if activation == "ReLU" else nn.Tanh()

    def forward(self, x):
        x = self.activation(self.input_layer(x))
        for layer in self.hidden_layers:
            x = self.activation(layer(x))
        return torch.cat([head(x) for head in self.heads], dim=1)  # raw logits

## Functions

In [8]:
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def _train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(loader)
    return avg_loss

def _calculate_val_acc(model, loader):
    global DEVICE

    model.eval()
    running_sum = 0.0
    total_samples = 0.0
    with torch.no_grad():
        for X,y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            output = model(X)
            pred = output.argmax(dim=1)
            correct = (pred == y).sum().item()
            running_sum += correct
            total_samples += float(y.size(0))

    acc = running_sum / total_samples

    return acc

def train_one_epoch(model, train_loader, val_loader, optimizer, criterion):
    global DEVICE

    train_loss = _train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss = _calculate_val_acc(model, val_loader)

    return train_loss, val_loss

def objective_func(trial, train_set, val_set,):
    global DEVICE

    EPOCHS = 5
    batch_size = trial.suggest_int('batch_size', 512, 1024)
    lr = trial.suggest_float('lr', 1e-3, 0.5)
    n_hidden = trial.suggest_int('n_hidden', 1, 25)
    hidden_dim = trial.suggest_int('hidden_dim', 1, 512)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

    model = IndependentHeadsMLP(input_dim=784, hidden_dim=hidden_dim, n_hidden=n_hidden, n_labels=10).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(EPOCHS):
        train_loss, val_acc = train_one_epoch(model,train_loader ,val_loader, optimizer, criterion)

    writer = SummaryWriter(f'runs/optuna_tensorboard/{trial.number}')

    # Logging
    hparams = {
        "lr": lr,
        "batch_size": batch_size,
        "n_hidden": n_hidden,
        "hidden_dim": hidden_dim
    }

    metrics = {
        "hparam/train_loss": train_loss,
        "hparam/val_acc": val_acc
    }

    writer.add_hparams(hparams, metrics)
    writer.flush()

    return val_acc


# Getting data and runing study

In [9]:
import optuna
# Using MNIST DataSet
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import numpy as np
from torch.utils.data import TensorDataset

mnist = fetch_openml('mnist_784', version=1, as_frame=False) # X: (70000, 784)
X, y = mnist["data"], mnist["target"]
X = X.astype(np.float32) / 255.0
y = y.astype(np.int64)
# scale to [0, 1]
X_trainval, X_test, y_trainval, y_test = train_test_split( X, y, test_size=5000, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=15000, random_state=42, stratify=y_trainval)

print(f"train size: {len(X_train)}")
print(f"val size: {len(X_val)}")
print(f"test size: {len(X_test)}")

# Data
X_train_t = torch.from_numpy(X_train).float()
y_train_t = torch.from_numpy(y_train).long()
train_set = TensorDataset(X_train_t, y_train_t)

x_val_t = torch.from_numpy(X_val).float()
y_val_t = torch.from_numpy(y_val).long()
val_set = TensorDataset(x_val_t, y_val_t)

study = optuna.create_study(direction='maximize')
study.optimize(lambda t: objective_func(t, train_set, val_set), n_trials=25, show_progress_bar=True)

[I 2026-04-13 13:20:59,588] A new study created in memory with name: no-name-a7b5065c-7180-4d4a-8e13-7bf9843ac298


train size: 50000
val size: 15000
test size: 5000


  0%|          | 0/25 [00:00<?, ?it/s]

[I 2026-04-13 13:21:13,632] Trial 0 finished with value: 0.1072 and parameters: {'batch_size': 520, 'lr': 0.2768936357476026, 'n_hidden': 23, 'hidden_dim': 315}. Best is trial 0 with value: 0.1072.
[I 2026-04-13 13:21:22,320] Trial 1 finished with value: 0.11253333333333333 and parameters: {'batch_size': 776, 'lr': 0.05894894822204615, 'n_hidden': 21, 'hidden_dim': 471}. Best is trial 1 with value: 0.11253333333333333.
[I 2026-04-13 13:21:37,679] Trial 2 finished with value: 0.11253333333333333 and parameters: {'batch_size': 522, 'lr': 0.018637762748147272, 'n_hidden': 24, 'hidden_dim': 34}. Best is trial 1 with value: 0.11253333333333333.
[I 2026-04-13 13:21:42,819] Trial 3 finished with value: 0.9686 and parameters: {'batch_size': 807, 'lr': 0.0016436549345249193, 'n_hidden': 3, 'hidden_dim': 251}. Best is trial 3 with value: 0.9686.
[I 2026-04-13 13:21:52,122] Trial 4 finished with value: 0.1042 and parameters: {'batch_size': 718, 'lr': 0.3025807339946828, 'n_hidden': 22, 'hidden_di

In [10]:
# RUN it to see training
# tensorboard --logdir Lab5/runs/optuna_tensorboard

# Export to CSV
is possible via TenserBoard Panel
```
HPARAMS -> TABLE VIEW -> CSV (left bottom corner)
```